In [ ]:
# NOTEBOOK NAME
# FeatureStatTimeChunks.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *

In [ ]:
FeatureStoragePath = '/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/22/20240228/QC2/V5/trackstats_20240228.000000_20240228.235500.nc'

FeatureXR = xr.open_dataset(FeatureStoragePath)

# add variables that rearrange times by time of day rather than time in a given feature's life
FeatureXR = AddFrameTimeVars(FeatureXR)

In [ ]:
FeatureXR

In [ ]:
np.min(np.array(FeatureXR['track_duration'] ))

In [ ]:
# CHAD FUNCTION
# CREATES ARRAYS FROM SPECIFIC VARIABLES IN FRAMETIMES
# USED TO LATER FIND MEANS AND SUCH

import math

def TimeChunkArray(variable: str, time_range: str, min_lifetime: int, ds) -> np.ndarray:
    """
    Extracts a numpy array from an xarray Dataset for a given variable,
    time range, and minimum track lifetime.

    Parameters
    ----------
    variable : str
        Name of the variable in the dataset (e.g., 'area_frametimes').
    time_range : str
        Time range in 'HH:MM-HH:MM' format (e.g., '14:00-16:00').
    min_lifetime : int
        Minimum lifetime in minutes a track must last to be included.
        Set to 0 to include all tracks.
    ds : xarray.Dataset
        The xarray Dataset to extract from.

    Returns
    -------
    np.ndarray
        Array of shape (n_tracks, n_frametimes) for the specified time window,
        filtered by minimum lifetime.
    """

    # --- Parse the time range string ---
    start_str, end_str = time_range.split('-')

    start_h, start_m = map(int, start_str.split(':'))
    end_h,   end_m   = map(int,   end_str.split(':'))

    start_minutes = start_h * 60 + start_m
    end_minutes   = end_h   * 60 + end_m

    # --- Convert minutes to FrameTime indices ---
    start_idx = start_minutes // 5
    end_idx   = (end_minutes  // 5) - 1

    # --- Compute minimum number of frames required ---
    # ceil(min_lifetime / 5) + 1 works universally:
    # 0 min -> 1 frame (all tracks qualify, since every track has >= 1 frame)
    # 5 min -> 2 frames, 6 min -> 3 frames, 10 min -> 3 frames, etc.
    min_frames = math.ceil(min_lifetime / 5) + 1

    # --- Filter tracks by lifetime ---
    track_mask = ds['track_duration'].values >= min_frames

    # --- Extract variable, filter tracks, return ---
    data = ds[variable].isel(FrameTimes=slice(start_idx, end_idx + 1))

    return data.isel(tracks=track_mask).values


In [ ]:
SampleArray = TimeChunkArray('area_frametimes', '14:00-16:00', 6, FeatureXR)

In [ ]:
np.nanmean(SampleArray)

In [ ]:
chunk_size = 30  # minutes
n_chunks = (24 * 60) // chunk_size  # e.g. 48 for 30-min chunks

MeanArea = np.full(n_chunks, np.nan)
MeanDepth = np.full(n_chunks, np.nan)
MeanIntensity = np.full(n_chunks, np.nan)

for i in range(n_chunks):
    start_minutes = i * chunk_size
    end_minutes   = start_minutes + chunk_size

    start_str = f"{start_minutes // 60:02d}:{start_minutes % 60:02d}"
    end_str   = f"{end_minutes   // 60:02d}:{end_minutes   % 60:02d}"
    time_range = f"{start_str}-{end_str}"

    MeanArea[i]      = np.nanmean( TimeChunkArray('area_frametimes', time_range, 6, FeatureXR) )
    MeanDepth[i]     = np.nanmean( TimeChunkArray('maxETH_20dbz_frametimes', time_range, 6, FeatureXR) )
    MeanIntensity[i] = np.nanmean( TimeChunkArray('max_dbz_frametimes', time_range, 6, FeatureXR) )


In [ ]:
MeanArea

In [ ]:
MeanDepth

In [ ]:
MeanIntensity

In [ ]:
# CHAD PLOT
# MEAN FEATURE CHARACTERISTICS OVER TIMES OF DAY

variables = {
    'area_frametimes'      : 'Mean Area',
    'max_dbz_frametimes' : 'Mean Intensity',
    'maxETH_20dbz_frametimes'     : 'Mean Depth',
}

results = {label: np.full(n_chunks, np.nan) for label in variables.values()}

# --- Compute means ---
for var, label in variables.items():
    for i in range(n_chunks):
        start_minutes = i * chunk_size
        end_minutes   = start_minutes + chunk_size

        start_str  = f"{start_minutes // 60:02d}:{start_minutes % 60:02d}"
        end_str    = f"{end_minutes   // 60:02d}:{end_minutes   % 60:02d}"
        time_range = f"{start_str}-{end_str}"

        SampleArray      = TimeChunkArray(var, time_range, 6, FeatureXR)
        results[label][i] = np.nanmean(SampleArray)

# --- Plot ---
ticks_per_hour = 60 // chunk_size
tick_positions = np.arange(0, n_chunks, ticks_per_hour)
tick_labels    = [f"{(i * chunk_size) // 60:02d}:{(i * chunk_size) % 60:02d}" for i in tick_positions]

for label, data in results.items():
    fig, ax = plt.subplots(figsize=(14, 5), facecolor='black')
    ax.set_facecolor('black')

    ax.plot(np.arange(n_chunks), data, color='white', linewidth=1)

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', color='white', fontsize=8)
    ax.set_xlim(0, n_chunks - 1)

    ax.yaxis.label.set_color('white')
    ax.xaxis.label.set_color('white')
    ax.tick_params(axis='y', colors='white')
    ax.spines['bottom'].set_color('white')
    ax.spines['left'].set_color('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlabel('Time of Day (UTC)', color='white')
    ax.set_ylabel(label, color='white')
    ax.set_title(f'{label} by Time of Day', color='white')

    plt.tight_layout()
    plt.show()


